# 01 · Data processing (ETL)

**Owner: Data Engineering.** Demonstrates reading NOAA-shaped records, station join, quality filtering and unit conversion. **Synthetic example; not NOAA observations.**

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("LivestockWeatherNotebook").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
from livestock_weather.settings import load_settings
cfg = load_settings(ROOT / "config" / "demo.json")


In [ ]:
from make_demo_data import create_demo
from livestock_weather.etl import read_stations, read_observations, clean_station_days
raw_dir = create_demo(ROOT / "data" / "demo_raw")
stations = read_stations(spark, raw_dir / "ghcnd-stations.txt")
raw = read_observations(spark, str(raw_dir / "*.csv"))
daily = clean_station_days(raw, stations)
print("US stations:", stations.count())
print("Clean station-days:", daily.count())
daily.select("station_id", "state", "date", "tmax_c", "tmin_c", "prcp_mm").show(12, False)

In [ ]:
output = ROOT / "data" / "demo_processed" / "clean_daily.parquet"
daily.write.mode("overwrite").partitionBy("year").parquet(str(output))
print("Saved:", output)